# Matrix Factorization

In [30]:
!pip install pyspark

In [35]:
!apt-get install -y openjdk-17-jdk-headless -qq

Selecting previously unselected package java-common.
(Reading database ... 125082 files and directories currently installed.)
Preparing to unpack .../java-common_0.72build2_all.deb ...
Unpacking java-common (0.72build2) ...
Selecting previously unselected package libpcsclite1:amd64.
Preparing to unpack .../libpcsclite1_1.9.5-3ubuntu1_amd64.deb ...
Unpacking libpcsclite1:amd64 (1.9.5-3ubuntu1) ...
Selecting previously unselected package openjdk-17-jre-headless:amd64.
Preparing to unpack .../openjdk-17-jre-headless_17.0.16+8~us1-0ubuntu1~22.04.1_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.16+8~us1-0ubuntu1~22.04.1) ...
Selecting previously unselected package ca-certificates-java.
Preparing to unpack .../ca-certificates-java_20190909ubuntu1.2_all.deb ...
Unpacking ca-certificates-java (20190909ubuntu1.2) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.16+8~us1-0ubuntu1~22.04.1_amd64.deb ...
Unp

In [36]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

In [38]:
import pandas as pd
import numpy as np

## Reading Ratings Data

In [82]:
ratings_df = pd.read_csv('https://raw.githubusercontent.com/manaranjanp/MLUL2/refs/heads/main/mf/u.data', sep = '\t')

In [83]:
ratings_df

,196,242,3,881250949
0,186,302,3,891717742
1,22,377,1,878887116
2,244,51,2,880606923
3,166,346,1,886397596
4,298,474,4,884182806
...,...,...,...,...
99994,880,476,3,880175444
99995,716,204,5,879795543
99996,276,1090,1,874795795
99997,13,225,2,882399156


In [84]:
ratings_df.columns = ['userId', 'itemId', 'rating', 'timestamp']

In [85]:
ratings_df

,userId,itemId,rating,timestamp
0,186,302,3,891717742
1,22,377,1,878887116
2,244,51,2,880606923
3,166,346,1,886397596
4,298,474,4,884182806
...,...,...,...,...
99994,880,476,3,880175444
99995,716,204,5,879795543
99996,276,1090,1,874795795
99997,13,225,2,882399156


In [86]:
ratings_df.drop('timestamp', axis = 1, inplace = True)

In [88]:
len(ratings_df.userId.unique())

943

In [89]:
len(ratings_df.itemId.unique())

1682

## Reading the movies metadata

In [105]:
movies_df = pd.read_csv('https://raw.githubusercontent.com/manaranjanp/MLUL2/refs/heads/main/mf/u.item',
                        encoding = 'iso-8859-1',
                        sep = '|',
                        header = None,
                        usecols=[0, 1])

In [106]:
movies_df

,0,1
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [107]:
movies_df.columns = ['itemId', 'name']

In [108]:
movies_df.head(10)

,itemId,name
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...
6,7,Twelve Monkeys (1995)
7,8,Babe (1995)
8,9,Dead Man Walking (1995)
9,10,Richard III (1995)


### Matrix Factorization Methods

In [95]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

def als_factorize_numpy(R, num_factors=3, reg_param=0.1, max_iter=20):
    """
    Perform matrix factorization using PySpark ALS on an incomplete NumPy matrix.

    Parameters
    ----------
    R : np.ndarray
        User-item matrix with NaN for missing entries.
    num_factors : int
        Number of latent features (rank of factorization).
    reg_param : float
        Regularization parameter (lambda).
    max_iter : int
        Number of ALS iterations.

    Returns
    -------
    W : np.ndarray
        User-feature matrix (num_users x num_factors)
    H : np.ndarray
        Item-feature matrix (num_items x num_factors)
    """

    # --- Step 1: Initialize Spark ---
    spark = SparkSession.builder \
        .appName("ALS Matrix Factorization") \
        .getOrCreate()

    ratings_df = spark.createDataFrame(R)

    # --- Step 3: Initialize ALS ---
    als = ALS(
        userCol="userId",
        itemCol="itemId",
        ratingCol="rating",
        rank=num_factors,
        regParam=reg_param,
        maxIter=max_iter,
        coldStartStrategy="drop",
        nonnegative=True
    )

    # --- Step 4: Train ALS model ---
    model = als.fit(ratings_df)

    # --- Step 5: Extract user and item factors ---
    user_factors = model.userFactors.toPandas().sort_values("id")
    item_factors = model.itemFactors.toPandas().sort_values("id")

    # Convert list of features to NumPy arrays
    W = np.vstack(user_factors["features"].values)
    H = np.vstack(item_factors["features"].values)

    # --- Step 6: Stop Spark session ---
    spark.stop()

    return W, H

## Factorizing User-Movies Ratings Matrix

In [97]:
num_factors = 20
reg_param=0.1
max_iter=20

W, H = als_factorize_numpy(ratings_df, num_factors, reg_param, max_iter)

print("W (User Feature Matrix):")
print(W)
print("\nH (Item Feature Matrix):")
print(H)

W (User Feature Matrix):
[[0.         1.12809873 0.31602931 ... 0.3486487  0.46158469 0.59807879]
 [0.         0.42957324 0.52321672 ... 0.35578695 0.         0.22145799]
 [1.18260074 0.43735939 0.57315671 ... 0.17884676 0.83707947 0.45343989]
 ...
 [0.08525906 1.05302763 0.16717716 ... 0.74179864 0.29981536 0.33975312]
 [0.22310804 0.24023417 0.         ... 0.43955851 0.40974984 0.28418434]
 [0.09160957 0.18461104 0.13570154 ... 1.11757135 0.66463262 0.        ]]

H (Item Feature Matrix):
[[0.29610994 0.43389085 0.20502549 ... 0.61629969 0.41589174 0.06748815]
 [0.         0.11262279 0.01116489 ... 0.49044728 0.245372   0.0938788 ]
 [0.34975588 0.71112168 0.         ... 0.28656828 0.67096698 0.88940191]
 ...
 [0.01618736 0.40003321 0.23520257 ... 0.25659353 0.36053029 0.        ]
 [0.         0.38879505 0.         ... 0.25925073 0.58345968 0.0272398 ]
 [0.1525715  0.63656938 0.24067582 ... 0.27317753 0.3398543  0.25638965]]


In [98]:
W.shape

(943, 20)

In [99]:
H.shape

(1682, 20)

## Finding Similarity

In [100]:
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import cosine, correlation

movies_sim = 1 - pairwise_distances( H, metric="cosine" )
movies_sim_df = pd.DataFrame( movies_sim )

In [111]:
def get_similar_movies( itemId, topN = 5 ):
    movieidx = movies_df[movies_df.itemId == itemId].index[0]
    movies_df['similarity'] = movies_sim_df.iloc[movieidx]
    top_n = movies_df.sort_values( ["similarity"], ascending = False )[0:topN]
    return top_n

In [102]:
movies_sim_df

,0,1,2,3,4,5,6,7,8,9,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
0,1.000000,0.879593,0.577651,0.825536,0.855614,0.695667,0.828334,0.926932,0.794129,0.773576,...,0.849719,0.856797,0.826595,0.826595,0.828263,0.673763,0.673763,0.673763,0.908079,0.819082
1,0.879593,1.000000,0.609233,0.793249,0.834773,0.676842,0.808102,0.835854,0.678337,0.681056,...,0.771493,0.793444,0.811613,0.811613,0.773658,0.620615,0.620615,0.620615,0.808231,0.755693
2,0.577651,0.609233,1.000000,0.745381,0.641873,0.602475,0.762018,0.516885,0.612646,0.652525,...,0.669685,0.661692,0.770313,0.770313,0.612226,0.638469,0.638469,0.638469,0.574285,0.759844
3,0.825536,0.793249,0.745381,1.000000,0.690239,0.666801,0.898415,0.850969,0.860809,0.784520,...,0.810495,0.868008,0.946037,0.946037,0.863025,0.818195,0.818195,0.818195,0.858389,0.874025
4,0.855614,0.834773,0.641873,0.690239,1.000000,0.639212,0.717170,0.740610,0.696305,0.613325,...,0.837959,0.804871,0.781595,0.781595,0.726443,0.549143,0.549143,0.549143,0.781279,0.755043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,0.673763,0.620615,0.638469,0.818195,0.549143,0.641762,0.767236,0.691237,0.716926,0.691077,...,0.703081,0.714567,0.744055,0.744055,0.732758,1.000000,1.000000,1.000000,0.726888,0.742455
1678,0.673763,0.620615,0.638469,0.818195,0.549143,0.641762,0.767236,0.691237,0.716926,0.691077,...,0.703081,0.714567,0.744055,0.744055,0.732758,1.000000,1.000000,1.000000,0.726888,0.742455
1679,0.673763,0.620615,0.638469,0.818195,0.549143,0.641762,0.767236,0.691237,0.716926,0.691077,...,0.703081,0.714567,0.744055,0.744055,0.732758,1.000000,1.000000,1.000000,0.726888,0.742455
1680,0.908079,0.808231,0.574285,0.858389,0.781279,0.602162,0.790603,0.867226,0.795112,0.730865,...,0.793157,0.841936,0.839347,0.839347,0.852551,0.726888,0.726888,0.726888,1.000000,0.824731


## Finding Similar Movies

In [109]:
movies_df[movies_df.itemId == 127]

,itemId,name
126,127,"Godfather, The (1972)"


In [112]:
get_similar_movies(127)

,itemId,name,similarity
126,127,"Godfather, The (1972)",1.000000
186,187,"Godfather: Part II, The (1974)",0.976505
522,523,Cool Hand Luke (1967),0.961979
520,521,"Deer Hunter, The (1978)",0.951103
1222,1223,King of the Hill (1993),0.951061


In [113]:
get_similar_movies(222)

,itemId,name,similarity
221,222,Star Trek: First Contact (1996),1.000000
469,470,Tombstone (1993),0.945380
227,228,Star Trek: The Wrath of Khan (1982),0.944127
226,227,Star Trek VI: The Undiscovered Country (1991),0.941774
1268,1269,Love in the Afternoon (1957),0.936156


In [114]:
get_similar_movies(118)

,itemId,name,similarity
117,118,Twister (1996),1.000000
120,121,Independence Day (ID4) (1996),0.963644
1144,1145,Blue Chips (1994),0.922906
545,546,Broken Arrow (1996),0.918480
754,755,Jumanji (1995),0.914640


In [115]:
get_similar_movies(465)

,itemId,name,similarity
464,465,"Jungle Book, The (1994)",1.000000
950,951,"Indian in the Cupboard, The (1995)",0.903339
999,1000,Lightning Jack (1994),0.902695
1538,1539,Being Human (1993),0.901678
417,418,Cinderella (1950),0.898564
